# Embeddings

Fuente: https://keras.io/examples/structured_data/collaborative_filtering_movielens/

In [ ]:
# ! pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 15.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

### Claude explanation (not from the professor)
Training hyperparameters: embedding size, Adam's learning rate, batch size and number of epochs.

**Note:** `batch_size` isn't used; the datasets below are batched with a hard-coded 32.

In [3]:
## hiperparámetros
base_embedding_dim = 64
learning_rate = 0.001
batch_size = 128
num_epochs = 3

### Claude explanation (not from the professor)
Loads all ratings. `/content/` is Colab's working folder; locally the path would be `datos/datos.db`.

In [4]:
con = sqlite3.connect('datos/datos.db')
ratings = pd.read_sql_query("SELECT id_libro, id_lector, rating FROM interacciones;", con)

### Claude explanation (not from the professor)
Vocabularies: the distinct readers and books, used to map each string ID to an embedding row.

**Note:** `item_vocabulary` is built from `id_lector` instead of `id_libro`, so books aren't in their own vocabulary and all map to the same out-of-vocabulary index. Also, `StringLookup` adds one out-of-vocabulary index, so the `Embedding` sizes should be `len(vocabulary) + 1`.

In [9]:
user_vocabulary = ratings["id_lector"].unique()
item_vocabulary = ratings["id_libro"].unique() # era id_lector, pero lo cambié a id_libro para que tenga sentido con el dataset de libros

### Claude explanation (not from the professor)
Shuffles, splits 80/20 into train and test, and separates the target (`rating`) from the inputs. Both are wrapped as `tf.data` datasets of dictionaries keyed by column name, so each column feeds the `Input` layer with the same name.

In [10]:
ratings = ratings.sample(frac=1, random_state=42)

X_train = ratings.sample(frac=0.8, random_state=42)
y_train = X_train.pop("rating")
X_test = ratings.drop(X_train.index)
y_test = X_test.pop("rating")

ds_train = tf.data.Dataset.from_tensor_slices((dict(X_train), y_train)).shuffle(buffer_size=len(ratings)).batch(32)
ds_test = tf.data.Dataset.from_tensor_slices((dict(X_test), y_test)).shuffle(buffer_size=len(ratings)).batch(32)

### Claude explanation (not from the professor)
The model, in the Keras functional API:
- Each ID goes through `StringLookup` (string → integer) and an `Embedding` (integer → 64-dimension learned vector).
- The **dot product** of the reader and book vectors is their affinity.
- `sigmoid × 10` maps it to the 0–10 rating range.

Compiled with Adam, MSE loss and MAE as the reported metric.

In [11]:
user_input = keras.layers.Input(name="id_lector", shape=(), dtype=tf.string)
user_embedding = keras.Sequential([
    keras.layers.StringLookup(vocabulary=user_vocabulary, mask_token=None),
    keras.layers.Embedding(
        input_dim=len(user_vocabulary),
        output_dim=base_embedding_dim)
], name="user_embedding")(user_input)

item_input = keras.layers.Input(name="id_libro", shape=(), dtype=tf.string)
item_embedding = keras.Sequential([
    keras.layers.StringLookup(vocabulary=item_vocabulary, mask_token=None),
    keras.layers.Embedding(
        input_dim=len(item_vocabulary),
        output_dim=base_embedding_dim)
], name="item_embedding")(item_input)

logits = keras.layers.Dot(axes=1, name="dot_similarity")([user_embedding, item_embedding])
prediction = keras.activations.sigmoid(logits) * 10

model = keras.Model(
    inputs=[user_input, item_input], outputs=prediction, name="baseline_model"
)

# model.summary()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
)

### Claude explanation (not from the professor)
Trains for `num_epochs`, evaluating on the test set after each epoch.

In [12]:
history = model.fit(
    ds_train,
    epochs=num_epochs,
    validation_data=ds_test,
)

Epoch 1/3
 4953/11536 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - loss: 6.9615 - mae: 2.2315

W0000 00:00:1792318694.646372 4596360 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: indices[12] = 10673 is not in [0, 10673)
	 [[{{function_node __inference_one_step_on_data_51773}}{{node baseline_model_1/user_embedding_1/embedding_2_1/GatherV2}}]]


InvalidArgumentError: Graph execution error:

Detected at node baseline_model_1/user_embedding_1/embedding_2_1/GatherV2 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/traitlets/config/application.py", line 1080, in launch_instance

  File "/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 768, in start

  File "/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/base_events.py", line 683, in run_forever

  File "/opt/homebr

### Claude explanation (not from the professor)
Plots training vs. test loss per epoch. If the test curve rises while the training curve keeps falling, the model is overfitting.

In [ ]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train", "eval"], loc="upper left")
plt.show()


ValueError: Invalid dtype: object